In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from housepriceprediction.config import DATA_RAW
from housepriceprediction.transformers import (
    ColumnDropper,
    FeatureEngineer,
    NeighborhoodMedianImputer,
    ConditionalImputer,
)
from housepriceprediction.features import ORDINAL_CATEGORIES

# 1. Load raw data and separate ID / Target
train_df = pd.read_csv(DATA_RAW / "train.csv", keep_default_na=False, na_values=["NA"])
X = train_df.drop(columns=["Id", "SalePrice"])

# 2. Sequentially apply steps 1-4 (implemented transformers)
X_trans = ColumnDropper().fit_transform(X)
X_trans = FeatureEngineer().fit_transform(X_trans)
X_trans = NeighborhoodMedianImputer().fit_transform(X_trans)
X_trans = ConditionalImputer().fit_transform(X_trans)

# 3. Separate into feature groups
ordinal_cols = list(ORDINAL_CATEGORIES.keys())

# Numeric: any column with numeric dtype after transformations (includes 'HasPool')
numeric_cols = X_trans.select_dtypes(include=["number", "int64", "float64"]).columns.tolist()

# Nominal: text/object columns that are NOT ordinal (includes 'MSSubClass' and 'MoSold')
text_cols = X_trans.select_dtypes(include=["object", "string"]).columns.tolist()
nominal_cols = [c for c in text_cols if c not in ordinal_cols]

print("NUMERIC_COLUMNS =", numeric_cols)
print("NOMINAL_COLUMNS =", nominal_cols)

# 4. Integrity check (Validate total coverage and uniqueness)
all_processed = set(X_trans.columns)
all_grouped = set(ordinal_cols) | set(numeric_cols) | set(nominal_cols)

print("\n--- Integrity Check ---")
print("Are all columns covered?", all_processed == all_grouped)
print("Missing columns:", all_processed - all_grouped)
print("Extra/duplicated columns:", all_grouped - all_processed)

# Intersections between groups (must be empty sets)
print("Intersection Numeric & Ordinal:", set(numeric_cols) & set(ordinal_cols))
print("Intersection Nominal & Ordinal:", set(nominal_cols) & set(ordinal_cols))
print("Intersection Numeric & Nominal:", set(numeric_cols) & set(nominal_cols))


NUMERIC_COLUMNS = ['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'MiscVal', 'YrSold', 'HasPool']
NOMINAL_COLUMNS = ['MSSubClass', 'MSZoning', 'Alley', 'LotShape', 'LandContour', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'BldgType', 'HouseStyle', 'RoofStyle', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'CentralAir', 'Electrical', 'GarageType', 'PavedDrive', 'Fence', 'MiscFeature', 'MoSold', 'SaleType', 'SaleCondition']

--- Integrity Check ---
Are all columns covered? True
Missing columns: set()
Extra/duplicated columns: set()
Intersection Numeric & Ordinal: set()
Intersection Nominal